In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ==========================================================
# Read Silver Tables
# ==========================================================

inpatient_df = spark.read.table(
    "healthcare_claims_catalog.silver.inpatient"
)

outpatient_df = spark.read.table(
    "healthcare_claims_catalog.silver.outpatient"
)

carrier_df = spark.read.table(
    "healthcare_claims_catalog.silver.carrier"
)

# ==========================================================
# Extract Facility Providers - Inpatient
# ==========================================================

inpatient_provider = inpatient_df.select(
    trim(col("PRVDR_NUM")).alias("PROVIDER_ID"),
    lit("Facility").alias("PROVIDER_TYPE")
)

# ==========================================================
# Extract Facility Providers - Outpatient
# ==========================================================

outpatient_provider = outpatient_df.select(
    trim(col("PRVDR_NUM")).alias("PROVIDER_ID"),
    lit("Facility").alias("PROVIDER_TYPE")
)

# ==========================================================
# Extract Physician Providers - Carrier
# ==========================================================

carrier_provider = carrier_df.select(
    trim(col("PRF_PHYSN_NPI_1").cast("string")).alias("PROVIDER_ID"),
    lit("Physician").alias("PROVIDER_TYPE")
)

# ==========================================================
# Combine All Providers
# ==========================================================

provider_df = (
    inpatient_provider
    .unionByName(outpatient_provider)
    .unionByName(carrier_provider)
)

# ==========================================================
# Remove Null / Blank Provider IDs
# ==========================================================

provider_df = provider_df.filter(
    col("PROVIDER_ID").isNotNull() &
    (trim(col("PROVIDER_ID")) != "")
)

# ==========================================================
# Remove Duplicate Providers
# ==========================================================

provider_df = provider_df.dropDuplicates(
    ["PROVIDER_ID", "PROVIDER_TYPE"]
)

# ==========================================================
# Generate Surrogate Key
# ==========================================================

window_spec = Window.orderBy(
    "PROVIDER_TYPE",
    "PROVIDER_ID"
)

provider_df = provider_df.withColumn(
    "PROVIDER_KEY",
    row_number().over(window_spec)
)

# ==========================================================
# Add Audit Column
# ==========================================================

provider_df = provider_df.withColumn(
    "gold_created_timestamp",
    current_timestamp()
)

# ==========================================================
# Final Column Order
# ==========================================================

dim_provider = provider_df.select(
    "PROVIDER_KEY",
    "PROVIDER_ID",
    "PROVIDER_TYPE",
    "gold_created_timestamp"
)

# ==========================================================
# Write Gold Table
# ==========================================================

dim_provider.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "healthcare_claims_catalog.gold.dim_provider"
    )

# ==========================================================
# Validation
# ==========================================================

print("=" * 60)
print("Gold Dimension - Provider Created Successfully")
print("=" * 60)

print("Total Providers :", dim_provider.count())

dim_provider.printSchema()

dim_provider.orderBy(
    "PROVIDER_KEY"
).show(20, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold Dimension - Provider Created Successfully
Total Providers : 620594
root
 |-- PROVIDER_KEY: integer (nullable = false)
 |-- PROVIDER_ID: string (nullable = true)
 |-- PROVIDER_TYPE: string (nullable = false)
 |-- gold_created_timestamp: timestamp (nullable = false)

+------------+-----------+-------------+--------------------------+
|PROVIDER_KEY|PROVIDER_ID|PROVIDER_TYPE|gold_created_timestamp    |
+------------+-----------+-------------+--------------------------+
|1           |01006H     |Facility     |2026-08-06 17:09:47.936858|
|2           |01006P     |Facility     |2026-08-06 17:09:47.936858|
|3           |01006V     |Facility     |2026-08-06 17:09:47.936858|
|4           |01008A     |Facility     |2026-08-06 17:09:47.936858|
|5           |01008J     |Facility     |2026-08-06 17:09:47.936858|
|6           |01008R     |Facility     |2026-08-06 17:09:47.936858|
|7           |0100AA     |Facility     |2026-08-06 17:09:47.936858|
|8           |0100AJ     |Facility     |2026-08-0